In [1]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 데이터 확인하기 2025.11.21
# 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# data loading
train, test = load_data()

In [3]:
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거
X_features['var3'] = X_features['var3'].replace(-999999, 2)
# var3 의 최소값 -99999 를 최빈값으로 변경하기

In [4]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features,
  y_labels,
)

In [5]:
# 2) Scaler 생성 (train에만 fit)
scaler = StandardScaler()
scaler.fit(X_train)

,copy,True
,with_mean,True
,with_std,True


In [6]:
# 3) train, val, test에 동일한 scaler 적용
X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [7]:
# 레이블의 분포 확인
cust_cnt = y_labels.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [8]:
class ThresholdModel:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        # 재학습 방지 — 이미 fit된 모델 그대로 사용
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)

In [15]:
# lgbm , logisticregression
lgbm_clf = LGBMClassifier(
    random_state      = 0,
    n_estimators      = 100,
    num_leaves        = 31,
    min_child_samples = 10,
    class_weight = {1:2}
)

lgbm_clf.fit(X_train, y_train)

pred = lgbm_clf.predict(X_val)
pred_proba = lgbm_clf.predict_proba(X_val)[:,1]

get_model_train_eval(
    lgbm_clf,
    'LightGBM_100_num31_min10_class1vs2',
    X_train, X_val,
    y_train, y_val
)

best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (pred_proba >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"\nBest Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# # 최적 threshold로 성능 출력
# pred_best = (pred_proba >= best_threshold).astype(int)
# get_clf_eval(y_val, pred_best, pred_proba)
threshold_model = ThresholdModel(lgbm_clf, best_threshold)

get_model_train_eval(
    threshold_model,
    "LightGBM_100_num31_min10_class1vs2_thr",
    X_train, X_val,
    y_train, y_val
)

[LightGBM] [Info] Number of positive: 2406, number of negative: 58410
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024703 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14383
[LightGBM] [Info] Number of data points in the train set: 60816, number of used features: 263
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.076113 -> initscore=-2.496374
[LightGBM] [Info] Start training from score -2.496374
[LightGBM] [Info] Number of positive: 2406, number of negative: 58410
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024682 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14383
[LightGBM] [Info] Number of data points in the train set: 60816, number of used features: 263
[LightGBM] [Info